# CICIDS2017 Intrusion Detector Lab1 
This notebook loads, cleans, trains, evaluates, and saves a binary intrusion detection workflow.

In [3]:
import random
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. Configure the experiment

In [ ]:
project_dir = Path.cwd()
data_dir = project_dir / "data"
# Keep full-data runs for final results; sampling is useful when checking code quickly.
USE_SAMPLE = False
SAMPLE_FRAC = 0.25
csv_files = sorted(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in data/")
for path in csv_files:
    print(" -", path.name)

Found 8 CSV files in data/
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


## 2. Load and clean the CSV files

In [ ]:
frames = []
for csv_path in csv_files:
    try:
        # UTF-8 is preferred; the fallback handles the older CICIDS2017 file with Latin-1 bytes.
        frame = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        frame = pd.read_csv(csv_path, encoding="latin1", low_memory=False)
        print(f"Used Latin-1 fallback for {csv_path.name}")
    # Normalize headers before looking for Label because some files contain leading spaces.
    frame.columns = [str(col).strip() for col in frame.columns]
    label_col = next((col for col in frame.columns if col.lower() == "label"), None)
    if frame.empty or label_col is None:
        print(f"Skipping {csv_path.name}: no usable label column")
        continue
    if label_col != "Label":
        frame = frame.rename(columns={label_col: "Label"})
    frames.append(frame)

if not frames:
    raise ValueError("No valid CICIDS2017 CSV files were found in the data folder.")
# Combine the valid files into one flow-level dataset for binary classification.
df = pd.concat(frames, ignore_index=True)
# Identifiers and timestamps can identify a capture rather than describe general traffic behavior.
metadata_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
df = df.drop(columns=[col for col in metadata_cols if col in df.columns], errors="ignore")
# Remove exact duplicates and convert infinite measurements to missing values.
df = df.drop_duplicates().replace([np.inf, -np.inf], np.nan)
for col in df.columns:
    if col != "Label":
        df[col] = pd.to_numeric(df[col], errors="coerce")
# The lab uses binary classification: benign traffic is 0 and every attack type is 1.
df["Label"] = df["Label"].astype(str).str.strip().apply(lambda label: 0 if label == "BENIGN" else 1)
if USE_SAMPLE:
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED)
print("Cleaned dataset shape:", df.shape)
print("Label distribution:")
print(df["Label"].value_counts().sort_index())

Used Latin-1 fallback for Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Cleaned dataset shape: (2813800, 81)
Label distribution:
Label
0    2263765
1     550035
Name: count, dtype: int64


## 3. Create stratified train, validation, and test sets

In [ ]:
X = df.drop(columns=["Label"])
y = df["Label"].astype(int)
# Stratification keeps the benign/attack ratio similar in all three partitions.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)
print("Train/validation/test sizes:", len(X_train), len(X_val), len(X_test))

Train/validation/test sizes: 1688280 562760 562760


## 4. Define evaluation metrics

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_proba=None):
    # FAR measures the proportion of benign flows incorrectly flagged as attacks.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "far": fp / (fp + tn) if (fp + tn) else 0.0,
        "roc_auc": roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan,
    }

## 5. Define training-only preprocessing and models

In [ ]:
# Each pipeline learns medians from X_train only, preventing validation/test leakage.
median_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
models = {
    "Logistic Regression": Pipeline([("imputer", median_imputer), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=SEED, class_weight="balanced"))]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight="balanced", n_jobs=-1))]),
    "MLP": Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler()), ("model", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=SEED))]),
}

## 6. Train and compare models on validation data

In [ ]:
validation_results = {}
for name, model in models.items():
    # Use validation F1 to choose a model before looking at final test performance.
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_proba = model.predict_proba(X_val)[:, 1]
    validation_results[name] = compute_binary_metrics(y_val, val_pred, val_proba)
best_model_name = max(validation_results, key=lambda key: validation_results[key]["f1"])
print("Best validation model by F1:", best_model_name)
print(pd.DataFrame(validation_results).T.sort_values("f1", ascending=False))

Best validation model by F1: Random Forest
                     accuracy  precision    recall        f1       far  \
Random Forest        0.999593   0.998601  0.999318  0.998960  0.000340   
MLP                  0.996229   0.986622  0.994191  0.990392  0.003276   
Logistic Regression  0.919276   0.718003  0.966729  0.824006  0.092253   

                      roc_auc  
Random Forest        0.999986  
MLP                  0.999873  
Logistic Regression  0.980116  


## 7. Evaluate models on the held-out test set

In [ ]:
# The test set remains untouched until this final evaluation stage.
test_results = {}
for name, model in models.items():
    test_pred = model.predict(X_test)
    test_proba = model.predict_proba(X_test)[:, 1]
    test_results[name] = compute_binary_metrics(y_test, test_pred, test_proba)
summary = pd.DataFrame(test_results).T.sort_values("f1", ascending=False)
print(summary)

                     accuracy  precision    recall        f1       far  \
Random Forest        0.999606   0.998601  0.999382  0.998991  0.000340   
MLP                  0.996261   0.986668  0.994309  0.990474  0.003264   
Logistic Regression  0.919420   0.718437  0.966602  0.824245  0.092044   

                      roc_auc  
Random Forest        0.999987  
MLP                  0.999876  
Logistic Regression  0.979783  


## 8. Run the scaling ablation study

In [ ]:
# Compare logistic regression with and without scaling while keeping imputation identical.
logreg_noscale = Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", LogisticRegression(max_iter=2000, random_state=SEED, class_weight="balanced"))])
logreg_noscale.fit(X_train, y_train)
logreg_no_scale_metrics = compute_binary_metrics(y_val, logreg_noscale.predict(X_val))
logreg_scaled = Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=SEED, class_weight="balanced"))])
logreg_scaled.fit(X_train, y_train)
logreg_scaled_metrics = compute_binary_metrics(y_val, logreg_scaled.predict(X_val))
print("Without scaling:", logreg_no_scale_metrics)
print("With scaling:", logreg_scaled_metrics)

Without scaling: {'accuracy': 0.8512651929774682, 'precision': 0.5999012570734116, 'recall': 0.7179543119983274, 'f1': 0.6536402082247106, 'far': np.float64(0.11634379010188778), 'roc_auc': nan}
With scaling: {'accuracy': 0.9192764233420997, 'precision': 0.7180029031495797, 'recall': 0.9667293899479124, 'f1': 0.824005702729717, 'far': np.float64(0.09225339202611578), 'roc_auc': nan}


## 9. Save feature importance and metrics

In [ ]:
# Read feature importance from the fitted Random Forest inside its preprocessing pipeline.
rf_model = models["Random Forest"].named_steps["model"]
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(feature_importance.head(10))
# Save artifacts separately so results can be inspected without rerunning training.
fig_dir = project_dir / "figures"
fig_dir.mkdir(exist_ok=True)
feature_importance.head(15).sort_values().plot(kind="barh", figsize=(10, 6), color="steelblue")
plt.title("Top 15 features for CICIDS2017 attack detection")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(fig_dir / "top_features.png", dpi=150)
plt.close()
results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)
summary.to_csv(results_dir / "model_metrics.csv")
print("Saved metrics to:", results_dir / "model_metrics.csv")
print("Saved feature plot to:", fig_dir / "top_features.png")

Destination Port           0.059619
Average Packet Size        0.051085
Source Port                0.049661
Packet Length Std          0.049638
Init_Win_bytes_forward     0.043998
Packet Length Variance     0.042860
Init_Win_bytes_backward    0.039898
Min Packet Length          0.034337
Max Packet Length          0.031968
Fwd Packet Length Min      0.031044
dtype: float64
Saved metrics to: /Users/nishantgandhi/Library/CloudStorage/OneDrive-LuleåUniversityofTechnology/D7084E AI For Cybersecurity XAI/Lab1/results/model_metrics.csv
Saved feature plot to: /Users/nishantgandhi/Library/CloudStorage/OneDrive-LuleåUniversityofTechnology/D7084E AI For Cybersecurity XAI/Lab1/figures/top_features.png
